# Create fishtank files

For `lineage_tracing/lineage`, which is analyzed with fishtank instead of
MERlin (this replaces `create_merlin_files.ipynb`). Writes fishtank's configs
to `metadata/`: `color_usage`, a one-row `color_usage_{SAMPLE_NAME}_mf.csv`
(the merfish acquisition's DAPI round, for cross-modality registration) and
`decoding_strategy`. Then builds `SAMPLE_DIR/fishtank/`: folders, reference
files and run scripts (cellpose, detect-spots, decode-spots, mosaics). Run
after `before_imaging` 02b (positions).

The lineage and merfish acquisitions of one sample are sibling folders:
```
<sample_id>/
  merfish/   <- MERFISH_SAMPLE_DIR
  lineage/   <- SAMPLE_DIR (this notebook)
    fishtank/
```
`IMAGING_DIR` is detected from the folders: `"lineage"` when split, `""` when
both acquisitions share one folder (then `MERFISH_SAMPLE_DIR` is
`SAMPLE_DIR`). Each side's positions file carries its own tag, so they never
collide.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/create_submit_files/)
SAMPLE_DIR   = MERCI_DIR.parent
METADATA_DIR = SAMPLE_DIR / "metadata"
FISHTANK_DIR = SAMPLE_DIR / "fishtank"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.dave import count_positions
from MERci.acquisition.fishtank_config import (
    create_color_usage_csv, create_decoding_strategy_csv,
    create_fishtank_folder_skeleton, copy_fishtank_reference_files,
    FishtankScriptsSpec, create_fishtank_scripts, cellpose_anisotropy,
    copy_fishtank_microscope_file,
)
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.display        import display_file
from MERci.acquisition.pipeline_config import load_pipeline_config

PIPELINE_ID     = "lineage_tracing_lineage"   # EDIT ME -- one of the ids in pipeline_export.PIPELINES
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / f"{PIPELINE_ID}_pipeline.yaml")

# SAMPLE_NAME is shared by both acquisitions.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
LINEAGE_LIB_VERSION = PIPELINE_CONFIG.fishtank.lineage_lib_version

# IMAGING_DIR: "" = one shared folder, "lineage" when split. Override only if
# it was detected wrong:
# IMAGING_DIR = "lineage"
MERFISH_SAMPLE_DIR = (SAMPLE_DIR.parent / "merfish") if IMAGING_DIR else SAMPLE_DIR

# Each acquisition's positions-file tag (empty in the shared-folder layout).
POSITIONS_TAG         = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
MERFISH_IMAGING_DIR   = "merfish" if IMAGING_DIR else ""
MERFISH_POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, MERFISH_IMAGING_DIR)

print(f"SAMPLE_DIR         : {SAMPLE_DIR}")
print(f"SAMPLE_NAME        : {SAMPLE_NAME}")
print(f"IMAGING_DIR        : {IMAGING_DIR!r}")
print(f"POSITIONS_TAG      : {POSITIONS_TAG}")
print(f"MERFISH_SAMPLE_DIR : {MERFISH_SAMPLE_DIR}")
print(f"MERFISH_POSITIONS_TAG : {MERFISH_POSITIONS_TAG}")
if not MERFISH_SAMPLE_DIR.is_dir():
    print(f"WARNING: {MERFISH_SAMPLE_DIR} does not exist -- set it manually if your "
          f"sibling acquisition folder is named differently.")

## Color usage + decoding strategy

fishtank's own config files, written to `metadata/` and read by the run scripts generated below.

### Round-tag mapping (manual)

One row per imaging series: each color's label (`"r1"`, `"beads"`, `"DAPI"`,
`"empty"`, or blank = not imaged). This is a protocol choice, not derived from
`round_info.csv`. `series` is a `{fov:03d}`-style path relative to the
scripts' `--input` root (default `SAMPLE_DIR/data`, e.g. `hybs/H01/....tif`).
`frames` is that series' frame-table CSV (`metadata/frame-table-*.csv`).

In [ ]:
COLOR_USAGE_COLORS = tuple(PIPELINE_CONFIG.fishtank.color_usage_colors)

COLOR_USAGE_ROWS = [
    # {"series": "data/....tif", "frames": str(METADATA_DIR / "frame-table-....csv"),
    #  "650": "r1", "560": "r2", "488": "beads", "405": "DAPI"},
]

if not COLOR_USAGE_ROWS:
    print("WARNING: COLOR_USAGE_ROWS is empty -- fill in the round-tag mapping above.")

color_usage_path = METADATA_DIR / f"color_usage_{SAMPLE_NAME}.csv"
create_color_usage_csv(color_usage_path, COLOR_USAGE_ROWS, colors=COLOR_USAGE_COLORS)
print(f"Saved: {color_usage_path}")

### Cross-modality color usage (merfish reference round)

One row naming the merfish acquisition's DAPI round, used by
`cellpose_ft_mf.slurm` to segment its nuclei for cross-modality registration.

In [3]:
COLOR_USAGE_ROWS_MF = [
    # {"series": "data/....tif", "frames": "...",
    #  "650": "", "560": "", "488": "beads", "405": "DAPI"},
]

color_usage_mf_path = METADATA_DIR / f"color_usage_{SAMPLE_NAME}_mf.csv"
create_color_usage_csv(color_usage_mf_path, COLOR_USAGE_ROWS_MF, colors=COLOR_USAGE_COLORS)
print(f"Saved: {color_usage_mf_path}")

Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\metadata\color_usage_251225_LT027_saving_time_mf.csv


### Decoding strategy

One row per decode target (e.g. `intBC` plus any base-editing targets).
`reference_file`/`whitelist` point into `fishtank/reference/`. Set
`FISHTANK_CLUSTER_DIR` to where `fishtank/` will be on the cluster.

In [ ]:
FISHTANK_CLUSTER_DIR = f"/n/CHANGE_ME/PROJECT/experiments/{SAMPLE_NAME}/fishtank"   # edit me

# Decode targets from pipeline.yaml (fishtank.targets), joined with
# FISHTANK_CLUSTER_DIR.
TARGETS = [
    (t.name, t.method,
     f"{FISHTANK_CLUSTER_DIR}/reference/{t.reference_file}" if t.reference_file else "",
     f"{FISHTANK_CLUSTER_DIR}/reference/{t.whitelist}" if t.whitelist else "")
    for t in PIPELINE_CONFIG.fishtank.targets
]

decoding_strategy_path = METADATA_DIR / f"decoding_strategy_{SAMPLE_NAME}.csv"
create_decoding_strategy_csv(decoding_strategy_path, TARGETS)
print(f"Saved: {decoding_strategy_path}")

## Folder skeleton + shared reference files

In [2]:
create_fishtank_folder_skeleton(FISHTANK_DIR, MERCI_DIR)
copy_fishtank_reference_files(LINEAGE_LIB_VERSION, MERCI_DIR, FISHTANK_DIR)
print(f"fishtank/ ready at {FISHTANK_DIR}")

fishtank/ ready at c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank


## Copy the merfish positions file alongside the lineage one

`generate_mosaics.slurm` reads both positions files from `SAMPLE_DIR/positions/`.
This copies the merfish one in, under its own name (a no-op when both share
one folder).

In [ ]:
LINEAGE_POSITIONS_NAME     = f"positions_{POSITIONS_TAG}.txt"
MERFISH_POSITIONS_SRC_NAME = f"positions_{MERFISH_POSITIONS_TAG}.txt"
MERFISH_POSITIONS_NAME     = MERFISH_POSITIONS_SRC_NAME

lineage_positions_path = SAMPLE_DIR / "positions" / LINEAGE_POSITIONS_NAME
merfish_positions_src  = MERFISH_SAMPLE_DIR / "positions" / MERFISH_POSITIONS_SRC_NAME
merfish_positions_dst  = SAMPLE_DIR / "positions" / MERFISH_POSITIONS_NAME

if not lineage_positions_path.exists():
    print(f"WARNING: {lineage_positions_path} not found -- run before_imaging notebook 02 first.")
if merfish_positions_src.exists():
    if merfish_positions_src.resolve() == merfish_positions_dst.resolve():
        print(f"merfish positions already in place (flat layout): {merfish_positions_dst}")
    else:
        shutil.copy2(merfish_positions_src, merfish_positions_dst)
        print(f"Copied merfish positions: {merfish_positions_dst}")
else:
    print(f"WARNING: {merfish_positions_src} not found -- copy the merfish acquisition's "
          f"positions file to {merfish_positions_dst} manually before running generate_mosaics.slurm.")

## Script parameters

`FishtankScriptsSpec`: every field can be overridden; defaults in
`fishtank_config.py`. FOV counts come from each positions file. `REF_SERIES`
must be one of the `"beads"` rows in `COLOR_USAGE_ROWS`.

**Microscope settings.** HAL's image XML has the wrong orientation and pixel
size on ST2, so every command gets `--microscope ../params/<scope>.yaml` (from
`MERci/data/configs/fishtank/microscope/`), and no `--scale_factor`, which
would override its pixel size.

**On-the-fly detection** (`on_the_fly: "True"`): each pass detects spots in
the rounds already on disk and skips the rest. Resubmit
`detect_spots_ft.slurm` as rounds arrive. The first pass needs ~50 GB (the
script requests 64gb); later passes ~8 GB, so submit them with `--mem=10G`.

**`max_drift`** (px) must exceed the real bead drift from the reference round,
or fishtank sets the drift to 0. ~100 px has been seen, so 300.

In [ ]:
N_FOVS_LINEAGE = count_positions(lineage_positions_path) if lineage_positions_path.exists() else 0
N_FOVS_MERFISH = count_positions(merfish_positions_src) if merfish_positions_src.exists() else 0
print(f"N_FOVS_LINEAGE = {N_FOVS_LINEAGE}")
print(f"N_FOVS_MERFISH = {N_FOVS_MERFISH}")

REF_SERIES = ""   # e.g. "data/....tif" -- a beads series from COLOR_USAGE_ROWS

# 3D cellpose: z step relative to the full-resolution pixel size, from each
# acquisition's DAPI frame table.
def dapi_anisotropy(rows, microscope, objective):
    dapi = [(row, color) for row in rows for color in COLOR_USAGE_COLORS if row.get(color) == "DAPI"]
    if not dapi:
        print("WARNING: no DAPI row -- cellpose runs without --anisotropy (z treated like xy).")
        return None
    row, color = dapi[0]
    return cellpose_anisotropy(row["frames"], float(color), microscope, objective)

MERFISH_PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / "lineage_tracing_merfish_pipeline.yaml")

# fishtank --microscope YAML for each acquisition's scope, copied into fishtank/params/.
def microscope_file(config):
    path = copy_fishtank_microscope_file(config.microscope, config.objective, MERCI_DIR, FISHTANK_DIR)
    return f"../params/{path.name}"

MICROSCOPE_LINEAGE = microscope_file(PIPELINE_CONFIG)
MICROSCOPE_MERFISH = microscope_file(MERFISH_PIPELINE_CONFIG)
print(f"microscope: lineage {MICROSCOPE_LINEAGE}, merfish {MICROSCOPE_MERFISH}")

ANISOTROPY_LINEAGE = dapi_anisotropy(COLOR_USAGE_ROWS, PIPELINE_CONFIG.microscope, PIPELINE_CONFIG.objective)
ANISOTROPY_MERFISH = dapi_anisotropy(COLOR_USAGE_ROWS_MF, MERFISH_PIPELINE_CONFIG.microscope,
                                     MERFISH_PIPELINE_CONFIG.objective)
print(f"anisotropy: lineage {ANISOTROPY_LINEAGE}, merfish {ANISOTROPY_MERFISH}")

spec = FishtankScriptsSpec(
    n_fovs_lineage = N_FOVS_LINEAGE,
    n_fovs_merfish = N_FOVS_MERFISH,
    cellpose_lineage = {"anisotropy": ANISOTROPY_LINEAGE, "microscope": MICROSCOPE_LINEAGE},
    cellpose_merfish = {"anisotropy": ANISOTROPY_MERFISH, "microscope": MICROSCOPE_MERFISH},
    detect_spots     = {"microscope": MICROSCOPE_LINEAGE,
                        "on_the_fly": "True",   # see the markdown above
                        "max_drift":  300,      # px, see the markdown above
                        "mem":        "64gb"},  # first pass; later passes: --mem=10G
    mosaic_lineage   = {"microscope": MICROSCOPE_LINEAGE},
    mosaic_merfish   = {"microscope": MICROSCOPE_MERFISH},
)

## Generate the scripts

In [5]:
# --input root: COLOR_USAGE_ROWS series are relative to it. For rounds on several
# drives, use a staging folder of symlinks (e.g. hybs/H03 -> the other drive).
LINEAGE_INPUT = str(SAMPLE_DIR / "data")

written = create_fishtank_scripts(
    spec, FISHTANK_DIR,
    experiment_label      = SAMPLE_NAME,
    lineage_input         = LINEAGE_INPUT,
    merfish_input          = str(MERFISH_SAMPLE_DIR / "data"),
    mosaic_lineage_input   = str(SAMPLE_DIR / "data"),
    mosaic_merfish_input   = str(MERFISH_SAMPLE_DIR / "data"),
    lineage_positions      = f"../../positions/{LINEAGE_POSITIONS_NAME}",
    merfish_positions      = f"../../positions/{MERFISH_POSITIONS_NAME}",
    color_usage_lineage    = f"../../metadata/color_usage_{SAMPLE_NAME}.csv",
    color_usage_merfish    = f"../../metadata/color_usage_{SAMPLE_NAME}_mf.csv",
    decoding_strategy      = f"../../metadata/decoding_strategy_{SAMPLE_NAME}.csv",
    ref_series             = REF_SERIES,
)

for name, path in written.items():
    print(f"Saved: {path}")
    display_file(path)

Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank\scripts\cellpose_ft.slurm
Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank\scripts\cellpose_ft_mf.slurm
Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank\scripts\detect_spots_ft.slurm
Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank\scripts\decode_spots_ft.slurm
Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\fishtank\scripts\generate_mosaics.slurm
